In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import pickle

# ================= 配置区域 =================
DATA_DIR = Path("data/ToM_Data/Human_fMRI_Data")
OUTPUT_DIR = Path("data/results_supcon_fixed")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

SUBJECTS = [f"TOM{i:03d}" for i in range(1, 22) if i != 6]
CONDITIONS = ["FB Prot Wrong", "FB Prot Corr", "FB Phys Wrong", "FB Phys Corr", "TB Prot Wrong", "TB Prot Corr", "TB Phys Wrong", "TB Phys Corr"]
PERIODS = ["Belief_Period_H5", "Perspective_Period_H5"]
ROIS = ["Left_PFC.h5", "Left_TPJ.h5", "Right_PFC.h5", "Right_TPJ.h5"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ================= 1. 模型定义 =================
class ContrastiveEncoder(nn.Module):

    def __init__(self, input_dim, hidden_dim=256, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x):
        feat = self.encoder(x)
        proj = self.projector(feat)
        return feat, F.normalize(proj, dim=1)


class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        anchor_dot_contrast = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-8)

        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-8)
        loss = -mean_log_prob_pos.mean()
        return loss


In [2]:
# ================= 2. 数据处理函数 =================
def load_and_preprocess_subject(sub, period_name, roi_file):
    input_path = DATA_DIR / period_name / sub / roi_file
    raw_trials = []
    labels = []

    try:
        with h5py.File(input_path, 'r') as f:
            for cond_name in f.keys():
                for trial_name in f[cond_name].keys():
                    group = f[cond_name][trial_name]
                    signal = group['Signal'][()]
                    label = int(group['Condition_Number'][()])

                    if signal.shape[0] >= 4:
                        # 1. 提取时间点 [4s, 6s]
                        valid_signal = signal[-2:, :]  # (2, n_voxels)
                        valid_signal = np.nan_to_num(valid_signal)
                        raw_trials.append(valid_signal)
                        labels.append(label)
    except Exception as e:
        return None, None

    if len(raw_trials) == 0:
        return None, None

    # Stack: (n_trials, 2, n_voxels)
    X_stack = np.array(raw_trials)
    n_trials, n_time, n_voxels = X_stack.shape

    # 2. 直接 Flatten，不做任何个体的聚类
    # 这样保留了 MNI 空间的解剖对齐
    # 形状变为: (n_trials, 2 * n_voxels)
    # 前一半特征是4s时的全脑图，后一半是6s时的全脑图
    X_flat = X_stack.reshape(n_trials, -1)

    # 3. Subject Centering (去中心化)
    # 减去该被试的平均模式，消除个体基线差异
    subject_mean = np.mean(X_flat, axis=0)
    X_centered = X_flat - subject_mean

    return X_centered, np.array(labels)


#遍历所有被试，并将它们堆叠
def prepare_group_data(period, roi):
    print(f"\nProcessing {period} / {roi}...")
    X_group_list = []
    y_group_list = []

    # 先读取一个被试确定 feature 维度，用于对齐检查
    for sub in tqdm(SUBJECTS, desc="Subjects"):
        X_sub, y_sub = load_and_preprocess_subject(sub, period, roi)
        if X_sub is not None:
            # 再次标准化，确保所有人方差一致
            scaler = StandardScaler()
            X_sub_norm = scaler.fit_transform(X_sub)

            X_group_list.append(X_sub_norm)
            y_group_list.append(y_sub)

    X_final = np.vstack(X_group_list)
    y_final = np.concatenate(y_group_list)
    return X_final, y_final


In [3]:
# ================= 3. 训练与提取函数 =================
def train_and_extract(X, y, task_name):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)

    model = ContrastiveEncoder(input_dim=X.shape[1]).to(DEVICE)
    criterion = SupConLoss(temperature=0.1)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    epochs = 50
    loss_history = []
    model.train()

    print(f"Training Model on shape {X.shape}...")
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_x, batch_y in train_dl:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            feat, proj = model(batch_x)
            loss = criterion(proj, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_dl))

    model.eval()
    with torch.no_grad():
        full_tensor = torch.FloatTensor(X).to(DEVICE)
        embeddings, _ = model(full_tensor)
        embeddings = embeddings.cpu().numpy()

    return embeddings, loss_history


# ================= 4. 主流程 =================
final_embeddings = {}

for period in PERIODS:
    for roi in ROIS:
        task_id = f"{period}_{roi.replace('.h5', '')}"

        X_raw, y_raw = prepare_group_data(period, roi)

        if X_raw is None:
            continue

        emb, losses = train_and_extract(X_raw, y_raw, task_id)

        final_embeddings[task_id] = {"embeddings": emb, "labels": y_raw, "loss": losses}

        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(emb)

        plt.figure(figsize=(10, 6))
        sns.scatterplot(x=X_pca[:, 0],
                        y=X_pca[:, 1],
                        hue=[CONDITIONS[i] for i in y_raw],
                        style=['False Belief' if i < 4 else 'True Belief' for i in y_raw],
                        palette='tab10',
                        s=60,
                        alpha=0.8)
        plt.title(f"Fixed SupCon: {task_id}")
        plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
        plt.tight_layout()

        save_path = OUTPUT_DIR / f"{task_id}_pca.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Saved plot to {save_path}")

output_pkl = OUTPUT_DIR / "all_supcon_embeddings_fixed.pkl"
with open(output_pkl, 'wb') as f:
    pickle.dump(final_embeddings, f)

print(f"\nAll done! saved to {output_pkl}")



Processing Belief_Period_H5 / Left_PFC.h5...


Subjects: 100%|██████████| 20/20 [00:00<00:00, 43.34it/s]


Training Model on shape (1280, 8344)...
Saved plot to data/results_supcon_fixed/Belief_Period_H5_Left_PFC_pca.png

Processing Belief_Period_H5 / Left_TPJ.h5...


Subjects: 100%|██████████| 20/20 [00:00<00:00, 57.94it/s]


Training Model on shape (1280, 4462)...
Saved plot to data/results_supcon_fixed/Belief_Period_H5_Left_TPJ_pca.png

Processing Belief_Period_H5 / Right_PFC.h5...


...

Training Model on shape (1280, 4462)...
Saved plot to data/results_supcon_fixed/Perspective_Period_H5_Left_TPJ_pca.png

Processing Perspective_Period_H5 / Right_PFC.h5...


Subjects: 100%|██████████| 20/20 [00:00<00:00, 49.19it/s]


Training Model on shape (1280, 8206)...
Saved plot to data/results_supcon_fixed/Perspective_Period_H5_Right_PFC_pca.png

Processing Perspective_Period_H5 / Right_TPJ.h5...


Subjects: 100%|██████████| 20/20 [00:00<00:00, 60.69it/s]


Training Model on shape (1280, 3454)...
Saved plot to data/results_supcon_fixed/Perspective_Period_H5_Right_TPJ_pca.png

All done! saved to data/results_supcon_fixed/all_supcon_embeddings_fixed.pkl
